# AMB 2D Simulation Sanity Check
This notebook verifies that the 2D Cahn-Hilliard active model simulation (Active Model B) integrates correctly and visualizes the **Mean Local EPR Density** over an ensemble.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import torch

from generate_trajectories import ActiveModelB

In [ ]:
# Core Simulation Parameters (Aligned with AMB_2D notebook)
kwargs = {
    'Lx': 64,              # Domain width
    'Ly': 64,              # Domain height
    'dx': 1.0,             # Spatial resolution
    'a': 0.25,             # Active model parameter A
    'b': 0.25,             # Active model parameter B
    'kappa': 4.0,          # Gradient coefficient
    'lam': 1.0,            # Active parameter (Lambda)
    'D': 0.1,              # Noise strength (Diffusivity)
    'dt': 0.001,           # Time step
    'smooth': False,       # Smoothing for stability
    'backend': 'torch',
    'use_gpu': torch.cuda.is_available()
}

print(f"Using Backend: {kwargs['backend']} | GPU Enabled: {kwargs['use_gpu']}")

model = ActiveModelB(**kwargs)

In [ ]:
# Sanity check parameters (shorter trajectory for quick verification)
n_seeds = 100
n_steps = 1000
burn_in = 5000

# Metric Evaluator (Mean EPR Density)
print(f'Simulating Ensemble ({n_seeds} seeds) and Computing EPR Density On-the-Fly...')
ensemble_epr_density = model.compute_mean_epr_on_the_fly(
    n_trajectories=n_seeds, n_steps=n_steps, burn_in=burn_in, show_progress=True
)

# Print Total Mean EPR
total_epr = np.sum(ensemble_epr_density) * (model.dx ** 2)
print(f'Total Mean EPR across ensemble: {total_epr:.6e}')

In [ ]:
# Visualization: 2D Heatmap & Cross-section
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Heatmap
ax = axes[0]
im = ax.imshow(
    ensemble_epr_density.T,
    origin='lower',
    aspect='equal',
    cmap='hot',
    extent=[0, kwargs['Lx']*kwargs['dx'], 0, kwargs['Ly']*kwargs['dx']]
)
fig.colorbar(im, ax=ax, label=r'$\langle\sigma\rangle$')
ax.set_title(f'Mean EPR Density Map ({n_seeds} seeds)')
ax.set_xlabel('x')
ax.set_ylabel('y')

# Cross-section through center
ax2 = axes[1]
center_y = kwargs['Ly'] // 2
x = np.arange(kwargs['Lx']) * kwargs['dx']
ax2.plot(x, ensemble_epr_density[:, center_y], color='crimson', lw=2, label=f'Profile at y={center_y}')
ax2.axhline(0, color='black', ls='--', lw=0.8)
ax2.set_title('EPR Profile Cross-section')
ax2.set_xlabel('Spatial Coordinate (x)')
ax2.set_ylabel(r'$\langle\sigma\rangle$')
ax2.legend()

plt.tight_layout()
plt.show()